# Netherland Single Cell Minimal Example

This notebook walks through a single timestep of the **netherland** salt marsh sediment model (based on Morris & Bowden 1986). The model simulates below-ground biomass and sediment dynamics inside a single `Cell`, which is a layered sediment column.

Each cell contains one or more `Layer` objects stacked from the bottom up. Depths within a cell are measured **downward from the current surface** (0 = marsh surface, positive = below surface).The cell also tracks its **absolute elevation** (i.e., cm above mean low water for example). This changes over time as sediment is deposited and decomposed.

In [ ]:
from pathlib import Path

import pandas as pd

from src import cell
from src import constants

## 1. Load Constants

Model parameters are stored in a TOML file (`data/morris_constants.toml`). `import_file()` returns a list of `Constants` objects; here we use the first entry.

Key parameters:
- `ro = 0.0105` g/cm² — below-ground biomass production rate at the surface
- `k = 0.7142` — labile decay rate (71% of labile sediment decomposes per year)
- `rd = 30.0` cm — root depth
- `du = 0.0`, `db = -30.0` → initial layer spans 0–30 cm below the surface

In [ ]:
K = constants.import_file()[0]
print(K)

Constants(id=1, sa=1.0, du=0.0, db=-30.0, bo=0.07182, bi=0.1, fo=0.83, k=0.7142, fc=0.2, ro=0.0105, rd=30.0, k1=0.1, k2=0.5, k3=0.0344, sv_to_ro=1.0, wa_to_rl=0.1, b=1.0)


## 2. Create the Initial Cell

`cell.factory(K)` constructs a `Cell` with a single 30 cm layer. Stocks in this layer:

- **biomass** — below-ground root biomass distributed by an exponential depth profile
- **labile** — easily decomposed organic sediment (see note below)
- **refractory** — recalcitrant organic sediment that resists decomposition
- **inorganic** — mineral sediment (silt, clay, sand)

All stock *lengths* (thickness in cm) sum to the layer depth. `top` and `bottom` are depths below the current surface.

> **Labile sediment** is organic material that bacteria and microbes break down easily — things
> like simple sugars, proteins, amino acids, and lipids from fresh plant material, algae, and
> phytoplankton. When it decomposes it releases nutrients (nitrogen, phosphorus), gases (CO₂,
> CH₄), and dissolved carbon. Because these products are mostly lost to the water column and
> atmosphere, the mass is removed from the sediment layer. A small fraction stays as living
> microbial biomass, but in this model that is treated as negligible.

In [ ]:
single_cell = cell.factory(K)
data = single_cell.write_data()
pd.DataFrame(data[0], columns=data[1])

   top  bottom  depth  biomass     labile  refractory  inorganic
0    0    30.0   30.0   1.3892  18.997571    4.749393   4.863836

## 3. Step the Cell Forward One Year

`step_forward(dep, top, yrs)` advances the cell by one timestep:

1. **Updates the existing layer** — biomass turns over; labile sediment decomposes (mass lost
   to CO₂, CH₄, nutrients); the layer *compacts* as its top depth increases.
2. **Adds a new top layer** — deposited sediment forms a fresh layer at depth `(0, dep)`
   relative to the new surface; new roots grow into it immediately.
3. **Shifts all older layers down** — depths are always relative to the *current* surface.
4. **Updates `Cell.elevation`** — absolute ground elevation changes by `dep − old_surface_depth`.

**Model choice — roots and deposited sediment:** roots occupy space *within* the deposited sediment. New layer sediment = `dep − bio.length`, so all stocks sum exactly to `dep`.

Parameters: `dep=1.0` cm deposition, `top=K.ro`, `yrs=1.0`

In [ ]:
stepped = single_cell.step_forward(dep=1.0, top=K.ro, yrs=1.0, sub_steps=1)
pd.DataFrame(stepped.write_data()[0], columns=data[1])

        top     bottom      depth   biomass    labile  refractory  inorganic
0  0.000000   1.000000   1.000000  0.139127  0.571620    0.142905   0.146348
1  1.000000  18.117384  17.117384  1.084017  6.220934    4.947250   4.854644

Row 0 is the **new top layer** (surface to 1 cm depth). Row 1 is the **original layer**, shifted down to start at 1 cm. The original layer shrank from 30 cm to ~17.1 cm: labile decomposition removed ~12.8 cm of sediment equivalent from the column.

New top layer stocks sum to 1.0 cm: 0.139 + 0.572 + 0.143 + 0.146 ≈ 1.0 ✓

## 4. Layer Elevations

`Cell.layer_elevations` converts the depth-based column view into **absolute elevation** units (same datum as `Cell.elevation`). Returns one value per layer boundary, surface-to-bottom:

    [surface_elevation, bottom_of_layer_0, bottom_of_layer_1, ...]

Each value = `cell.elevation − layer.bottom_depth`.

In [ ]:
print(f'Initial cell  elevation: {single_cell.elevation} cm')
print(f'Initial layer_elevations: {single_cell.layer_elevations}')
print()
elevs = [f'{v:.4f}' for v in stepped.layer_elevations]
print(f'After step    elevation: {stepped.elevation:.4f} cm')
print(f'Stepped layer_elevations: {elevs}')

Initial cell  elevation: 0.0 cm
Initial layer_elevations: [0.0, -30.0]

After step    elevation: -11.8826 cm
Stepped layer_elevations: ['-11.8826', '-12.8826', '-30.0000']


After one year the surface dropped from **0.0 cm to −11.88 cm** — a net loss of ~11.88 cm. This reflects the high labile decay rate (`k = 0.7142`: 71%/yr) applied to the full 30 cm initial layer, far outpacing 1 cm of new deposition.

The original layer bottom is fixed at **−30.0 cm**: the base of each layer does not move.